# `%sdi` + `%metadata` — SDI metadata into DDS

A test notebook for the `%sdi` and `%metadata` magics from `eea_datalakehouse.notebook.magics`.
`%sdi` downloads a dataset's ISO 19115-3 metadata record from the EEA SDI catalogue;
`%metadata` uploads it to the dataset's `metadata` folder in DDS — a **document upload**, never an ingest: nothing becomes a
Dremio table. See `docs/sdi-integration-plan.md` for the design.

**Kernel environment** — read the same way `%catalog` and `%ingest` read theirs, when each magic's
first call builds its session:

| Variable | Needed for | Notes |
|---|---|---|
| `SDI_API_URL` | `%sdi get_xml`, `%sdi resolve_series` | empty or unset → `https://sdi.eea.europa.eu/catalogue` |
| `SDI_USERNAME` / `SDI_PASSWORD` | non-public records only | optional |
| `DDS_BASE_URL` | `%metadata push_to_dds` | read from the first `.env` in the notebook's folder or a parent (here `debugger/.env`); else the kernel environment |
| `_DREMIO_USER` / `_DREMIO_PWD` | `%metadata push_to_dds` | as for `%ingest`; else `DREMIO_USERNAME` / `DREMIO_TOKEN`, as for `%catalog` |

`%sdi` makes public, read-only calls and needs none of the DDS settings. Install the extra this
needs once: `pip install "EEADataLakehouse[notebook]"`.

In [ ]:
import eea_datalakehouse.notebook  # registers %catalog/%ingest/%sdi — no %load_ext needed

## Quick reference

`%sdi help` and `%metadata help` render every command as an HTML table, the same format as
`%catalog help` and `%ingest help`. They work before any environment variable is set.

In [ ]:
%sdi help

In [ ]:
%metadata help

## Settings for this test

The bathing water release used throughout the library's tests. The DDS path is
**provisional** until the DDS document path convention is agreed; `%metadata push_to_dds` appends
`/metadata/<uuid>.xml` to it.

In [ ]:
RELEASE_UUID = "070d9baa-448d-4168-8514-7dadb3ad876d"   # Bathing Water Directive, 2025 v1.0
SERIES_UUID = "c3858959-90da-4c1b-b9ca-492db0e514df"    # its series
DDS_PATH = "catalog/water_management_resources/bathing_water/bwd"

## 1. Get the XML from SDI

Runs immediately. The record is checked before it is returned: it must be ISO 19115-3
(`mdb:MD_Metadata`) and its identifier must be `RELEASE_UUID`. Any failure prints one
`sdi error: ...` line instead of a traceback.

In [ ]:
metadata = %sdi get_xml(RELEASE_UUID)
metadata

What was parsed out of it — title, edition, scope and the metadata dates:

In [ ]:
metadata.record

In [ ]:
print(metadata.xml[:600].decode())

## 2. A series instead of a release

`resolve_series` returns the one release of a series that is not superseded. If there are zero
or several it refuses to guess and lists them all. The bathing water series currently has
**two** non-superseded releases, so expect an `sdi error` listing both — the cell after it
uses the release UUID directly, which is the normal way.

In [ ]:
%sdi resolve_series(SERIES_UUID)

## 3. Push it to DDS

**Writes to DDS.** `%metadata push_to_dds` uploads the record `%sdi get_xml` fetched last, so
only the DDS path is needed. It compares with the copy already there first:

| DDS already has | Result |
|---|---|
| nothing | `uploaded` |
| the same bytes | `unchanged` (no upload) |
| an older copy | `replaced` |
| a newer, same-dated or non-ISO copy (edited in DDS) | refused, unless `force=True` |

The DDS document endpoints are still interface assumptions to confirm with the DDS team, so
run this against a test instance first. Set `RUN_PUSH = True` to run it.

Check which DDS the push will go to first. `DDS_BASE_URL` is taken from the first `.env`
found in this notebook's folder or a parent (read once, when `%metadata` builds its session), and
from the kernel environment only if no `.env` sets it:

In [ ]:
%metadata dds_base_url()

In [ ]:
RUN_PUSH = False

if RUN_PUSH:
    result = %metadata push_to_dds(DDS_PATH)
    print(result)
else:
    from eea_datalakehouse.sdi import metadata_path
    print("RUN_PUSH is False — would upload to", metadata_path(DDS_PATH, metadata.uuid))

Running the push a second time should report `unchanged`. To replace a copy that was edited in
DDS, pass `force=True`; to push a record other than the last one, pass it explicitly:

```
%metadata push_to_dds(DDS_PATH, force=True)
%metadata push_to_dds(DDS_PATH, metadata=other_metadata)
```

## 4. Optional: record the provenance on the wiki

`provenance_tags()` gives `sdi_record_uuid` / `sdi_edition` / `sdi_date_stamp` in the shape
`Catalog.setmeta2wiki`'s `tags` take (a folder's wiki "# Meta Data" section). Writing them is
up to you — neither magic writes to Dremio itself:

```python
from eea_datalakehouse.catalog import Catalog

with Catalog(DREMIO_BASE_URL, DREMIO_TOKEN, username=DREMIO_USERNAME) as catalog:
    catalog.setmeta2wiki(
        "catalog.water_management_resources.bathing_water.bwd",
        tags=metadata.provenance_tags(),
        overwrite=True,   # merge with the tags already there
        idempotency_key=f"sdi-provenance-{metadata.uuid}",
    )
```

In [ ]:
metadata.provenance_tags()

## What errors look like

```
%sdi get_xml("00000000-0000-0000-0000-000000000000")
# -> sdi error: SDI API error 404 on GET /srv/api/records/.../formatters/xml: no SDI record with UUID '...'

%metadata push_to_dds(DDS_PATH)   # in a kernel with no Dremio identity
# -> metadata error: pushing to DDS needs a Dremio identity in the kernel environment: ...
```

In [ ]:
%sdi get_xml("00000000-0000-0000-0000-000000000000")